# LangGraph Pipeline Test — Direct Invocation (No Server)

Tests the core generation pipeline by invoking the LangGraph graph directly.
No HTTP server, no Docker — pure pipeline verification.

**Test Plan:**
1. Import & config verification
2. Smoke test — single topic, small batch (5 quiz)
3. Medium batch — 15 quiz (lesson scale)
4. Large batch — 30 quiz (Kahoot scale)
5. Multi game-type — quiz + flashcard + fill_blank
6. Multi-chapter topic — broad coverage
7. Difficulty levels — recall / comprehension / application / high_application
8. STEM subject coverage — Math, Physics, Chemistry
9. Quality metrics summary
10. Suspicious analysis — tự động tìm nguyên nhân và đề xuất giải pháp
11. Focus rerun — diff_high_application (nếu vẫn thấp)

**Rules (bổ sung):**
- Nếu phát hiện suspicious (pass rate thấp, iterations > 1, hoặc generated quá cao so với requested), notebook phải tự động phân tích nguyên nhân khả dĩ và in ra giải pháp khuyến nghị.
- Ưu tiên phân tích riêng `diff_high_application` trước các nhóm khác.

**Industry Benchmarks:**
- Kahoot: 20-30 candidate questions per topic
- Khan Academy: generate 2-3x needed, curate down
- Duolingo: ~200 candidates per lesson, show ~14
- ASEE best practice: generate 2-3x required, prune

## 1. Setup & Configuration

In [1]:
import sys, json, time
from pathlib import Path

project_root = str(Path.cwd().parent.parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.config import get_settings
from src.config.constants import (
    OVERSHOOT_BY_DIFFICULTY, REVIEW_THRESHOLD_BY_DIFFICULTY,
    YIELD_OVERSHOOT_RATIO,
)
from src.api.schemas import (
    GenerationRequest, GameType, DifficultyLevel, DocScope,
    GameContentResponse, QuizQuestion, Flashcard, FillBlankQuestion,
)
from src.graph.builder import compile_graph
from src.graph.state import MAX_REVIEW_ITERATIONS

settings = get_settings()
print(f'Project: {settings.gcp_project_id}')
print(f'Generation model: {settings.generation_model} @ {settings.generation_model_location or settings.gcp_location}')
print(f'Review model: {settings.review_model} @ {settings.review_model_location}')
print(f'Max review iterations: {MAX_REVIEW_ITERATIONS}')
print(f'Old overshoot (uniform): {YIELD_OVERSHOOT_RATIO}')
print(f'New overshoot (by difficulty): {OVERSHOOT_BY_DIFFICULTY}')
print(f'Review thresholds (by difficulty): {REVIEW_THRESHOLD_BY_DIFFICULTY}')

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project: green-mercury-485016-n1
Generation model: gemini-2.5-flash @ asia-southeast1
Review model: gemini-3.1-flash-lite-preview @ global
Max review iterations: 3
Old overshoot (uniform): 1.3
New overshoot (by difficulty): {'recall': 1.1, 'comprehension': 1.2, 'application': 1.4, 'high_application': 2.0}
Review thresholds (by difficulty): {'recall': 0.7, 'comprehension': 0.7, 'application': 0.65, 'high_application': 0.55}


In [2]:
# ── Helpers ──

def pp(data):
    if hasattr(data, 'model_dump'):
        data = data.model_dump()
    print(json.dumps(data, ensure_ascii=False, indent=2, default=str))

def make_request(
    topic: str,
    num_questions: int = 10,
    game_types: list[str] | None = None,
    difficulty: str = 'comprehension',
    doc_scope: str = 'system',
) -> GenerationRequest:
    gts = [GameType(g) for g in (game_types or ['quiz'])]
    return GenerationRequest(
        user_id='pipeline_test',
        topic=topic,
        game_types=gts,
        num_questions=num_questions,
        difficulty=DifficultyLevel(difficulty),
        doc_scope=DocScope(doc_scope),
    )

async def run_pipeline(request: GenerationRequest) -> dict:
    """Run pipeline with fresh compile (no singleton cache)."""
    app = compile_graph()
    initial_state = {
        'request': request,
        'doc_scope': request.doc_scope.value,
        'iteration_count': 0,
        'rejected_items': [],
        'reviewed_items': [],
        'errors': [],
    }
    t0 = time.time()
    result = await app.ainvoke(initial_state)
    elapsed = time.time() - t0
    return {**result, '_elapsed': elapsed}

def summarize(result: dict, label: str = ''):
    prefix = f'[{label}] ' if label else ''
    output: GameContentResponse | None = result.get('final_output')
    errors = result.get('errors', [])
    elapsed = result.get('_elapsed', 0)
    iterations = result.get('iteration_count', 0)
    rejected = result.get('rejected_items', [])

    if not output:
        print(f'{prefix}❌ No output. Errors: {errors}')
        return None

    c = output.content
    m = output.metadata
    qn, fn, bn = len(c.quiz), len(c.flashcard), len(c.fill_blank)
    total = qn + fn + bn

    print(f'{prefix}✅ {total} items in {elapsed:.1f}s ({iterations} iter)')
    print(f'  quiz={qn}  flashcard={fn}  fill_blank={bn}')
    print(f'  generated={m.total_generated}  passed={m.total_passed_review}  rejected={m.total_rejected}')
    if iterations > 1:
        print(f'  🔁 Feedback loop used: {iterations - 1} retry iteration(s)')
    if errors:
        print(f'  ⚠️ errors: {errors}')
    return {'quiz': qn, 'fc': fn, 'fb': bn, 'total': total, 'elapsed': elapsed,
            'generated': m.total_generated, 'passed': m.total_passed_review,
            'rejected': m.total_rejected, 'iterations': iterations}

ALL = {}  # accumulate results
print(f'DifficultyLevel values: {[d.value for d in DifficultyLevel]}')
print('Helpers ready ✓')

DifficultyLevel values: ['recall', 'comprehension', 'application', 'high_application']
Helpers ready ✓


## 2. Smoke Test — 5 Quiz Questions

In [3]:
req = make_request('Đạo hàm', num_questions=5, game_types=['quiz'])
res = await run_pipeline(req)
ALL['smoke_5q'] = summarize(res, 'Smoke 5Q')

out = res['final_output']
assert out is not None, 'No output'
assert len(out.content.quiz) >= 3, f'Expected >=3 quiz, got {len(out.content.quiz)}'

q = out.content.quiz[0]
assert len(q.options) == 4
assert any(o.is_correct for o in q.options)
assert q.question and q.explanation

print(f'\nSample: {q.question[:120]}')
print(f'Correct: option {q.correct_answer_index}')
print(f'Trace: {(q.computation_trace or "none")[:120]}')
print('\n✓ Smoke test PASSED')

2026-03-29 10:53:05 [info     ] creating_graph                
2026-03-29 10:53:05 [info     ] graph_created                 
2026-03-29 10:53:05 [info     ] graph_compiled_without_checkpointer
2026-03-29 10:53:05 [info     ] supervisor_analyzing           doc_scope=system topic='Đạo hàm' user_id=pipeline_test
2026-03-29 10:53:05 [info     ] creating_review_llm            location=global max_output_tokens=10 model=gemini-3.1-flash-lite-preview temperature=0
2026-03-29 10:53:05 [info     ] rate_limiter_init_bucket       capacity=5.0 refill_rate=1.0 rpm=60
2026-03-29 10:53:05 [info     ] rate_limiter_init_semaphore    max_concurrent=10
2026-03-29 10:53:07 [info     ] supervisor_classified          content_type=math
2026-03-29 10:53:07 [info     ] routing_to_agent               content_type=math route=math_agent
2026-03-29 10:53:07 [info     ] math_agent_starting            doc_scope=system iteration=0 num_questions=5 num_still_needed=5 num_to_generate=6 topic='Đạo hàm' user_id=pipeline_t

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-29 10:53:08 [debug    ] retrieved_documents            count=30
2026-03-29 10:53:08 [info     ] context_retrieved              chunks=30 sources=30
2026-03-29 10:53:09 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-29 10:53:09 [info     ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.7
2026-03-29 10:53:38 [info     ] math_agent_code_execution_done code_traces_count=6 generation_attempt=1 text_length=4560
2026-03-29 10:53:38 [info     ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.3


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:1076: UserWarning: 
        - 'executable_code': Always present.
        - 'execution_result' & 'image_url': May be absent for some queries.

        Validate before using in production.

  warnings.warn(


2026-03-29 10:53:53 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=6
2026-03-29 10:53:53 [info     ] math_agent_generated           items_count=6
2026-03-29 10:53:53 [info     ] reviewer_reviewing             items_count=6 iteration=0
2026-03-29 10:53:53 [info     ] creating_review_llm            location=global max_output_tokens=4096 model=gemini-3.1-flash-lite-preview temperature=0.0
2026-03-29 10:53:57 [info     ] reviewer_completed             iteration=0 passed=6 rejected=0
2026-03-29 10:53:57 [info     ] review_router_deciding         iteration=1 max_iterations=3 pass_rate=1.0 rejected=0 reviewed=6
2026-03-29 10:53:57 [info     ] review_router_pass             reason=sufficient_items
2026-03-29 10:53:57 [info     ] formatter_trimming             after=5 before=6
2026-03-29 10:53:57 [info     ] formatter_starting             game_types=['quiz'] items_count=5 user_id=pipeline_test
2026-03-29 10:54:06 [info     ] formatter_quiz_batch_parsed    batch_start=0 it

## 3. Medium Batch — 15 Quiz (Lesson Scale)

In [4]:
req = make_request('Phương trình bậc hai', num_questions=15, game_types=['quiz'])
res = await run_pipeline(req)
ALL['medium_15q'] = summarize(res, '15Q Quiz')

out = res['final_output']
assert out is not None
assert len(out.content.quiz) >= 10, f'Expected >=10, got {len(out.content.quiz)}'

questions = [q.question for q in out.content.quiz]
assert len(set(questions)) == len(questions), 'Duplicate questions!'
print(f'Unique questions: {len(set(questions))}/{len(questions)}')

print('\n✓ Medium batch PASSED')

2026-03-29 10:54:12 [info     ] creating_graph                
2026-03-29 10:54:12 [info     ] graph_created                 
2026-03-29 10:54:12 [info     ] graph_compiled_without_checkpointer
2026-03-29 10:54:12 [info     ] supervisor_analyzing           doc_scope=system topic='Phương trình bậc hai' user_id=pipeline_test
2026-03-29 10:54:15 [info     ] supervisor_classified          content_type=math
2026-03-29 10:54:15 [info     ] routing_to_agent               content_type=math route=math_agent
2026-03-29 10:54:15 [info     ] math_agent_starting            doc_scope=system iteration=0 num_questions=15 num_still_needed=15 num_to_generate=18 topic='Phương trình bậc hai' user_id=pipeline_test
2026-03-29 10:54:15 [info     ] retrieving_context             doc_scope=system query='Phương trình bậc hai comprehension level educational content' user_id=pipeline_test
2026-03-29 10:54:15 [debug    ] creating_retriever             doc_scope=system extractive_answers=True filter='user_id: ANY("

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-29 10:54:17 [debug    ] retrieved_documents            count=30
2026-03-29 10:54:17 [info     ] context_retrieved              chunks=30 sources=30
2026-03-29 10:54:17 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-29 10:57:01 [info     ] math_agent_code_execution_done code_traces_count=20 generation_attempt=1 text_length=8579


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:1076: UserWarning: 
        - 'executable_code': Always present.
        - 'execution_result' & 'image_url': May be absent for some queries.

        Validate before using in production.

  warnings.warn(


2026-03-29 10:57:10 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-29 10:57:23 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=7
2026-03-29 10:57:33 [info     ] math_agent_batch_parsed        attempt=1 batch=3 items=4
2026-03-29 10:57:33 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=18
2026-03-29 10:57:33 [info     ] math_agent_generated           items_count=18
2026-03-29 10:57:33 [info     ] reviewer_reviewing             items_count=18 iteration=0
2026-03-29 10:57:38 [info     ] reviewer_completed             iteration=0 passed=18 rejected=0
2026-03-29 10:57:38 [info     ] review_router_deciding         iteration=1 max_iterations=3 pass_rate=1.0 rejected=0 reviewed=18
2026-03-29 10:57:38 [info     ] review_router_pass             reason=sufficient_items
2026-03-29 10:57:38 [info     ] formatter_trimming             after=15 before=18
2026-03-29 10:57:38 [info     ] formatter_starting             game_types=['qu

## 4. Large Batch — 30 Quiz (Kahoot Scale)

Kahoot generates 20-30 candidate questions per topic.

In [5]:
req = make_request('Tích phân', num_questions=30, game_types=['quiz'])
res = await run_pipeline(req)
ALL['large_30q'] = summarize(res, '30Q Quiz')

out = res['final_output']
assert out is not None
n = len(out.content.quiz)
assert n >= 21, f'Expected >=21 (70% of 30), got {n}'

questions = [q.question for q in out.content.quiz]
print(f'Unique: {len(set(questions))}/{len(questions)}')

for idx in [0, n//2, -1]:
    q = out.content.quiz[idx]
    print(f'\n[{idx}] {q.question[:100]}')
    print(f'    topic: {q.topic}')

print('\n✓ Large batch PASSED')

2026-03-29 10:58:09 [info     ] creating_graph                
2026-03-29 10:58:09 [info     ] graph_created                 
2026-03-29 10:58:09 [info     ] graph_compiled_without_checkpointer
2026-03-29 10:58:09 [info     ] supervisor_analyzing           doc_scope=system topic='Tích phân' user_id=pipeline_test
2026-03-29 10:58:11 [info     ] supervisor_classified          content_type=math
2026-03-29 10:58:11 [info     ] routing_to_agent               content_type=math route=math_agent
2026-03-29 10:58:11 [info     ] math_agent_starting            doc_scope=system iteration=0 num_questions=30 num_still_needed=30 num_to_generate=36 topic='Tích phân' user_id=pipeline_test
2026-03-29 10:58:11 [info     ] retrieving_context             doc_scope=system query='Tích phân comprehension level educational content' user_id=pipeline_test
2026-03-29 10:58:11 [debug    ] creating_retriever             doc_scope=system extractive_answers=True filter='user_id: ANY("__system__")' search_engine_id=gp

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-29 10:58:13 [debug    ] retrieved_documents            count=29
2026-03-29 10:58:13 [info     ] context_retrieved              chunks=29 sources=29
2026-03-29 10:58:13 [info     ] math_agent_context_retrieved   context_chunks=29
2026-03-29 10:58:58 [info     ] math_agent_code_execution_done code_traces_count=3 generation_attempt=1 text_length=16345


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:1076: UserWarning: 
        - 'executable_code': Always present.
        - 'execution_result' & 'image_url': May be absent for some queries.

        Validate before using in production.

  warnings.warn(


2026-03-29 10:59:08 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-29 10:59:17 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=7
2026-03-29 10:59:27 [info     ] math_agent_batch_parsed        attempt=1 batch=3 items=7
2026-03-29 10:59:36 [info     ] math_agent_batch_parsed        attempt=1 batch=4 items=7
2026-03-29 10:59:43 [info     ] math_agent_batch_parsed        attempt=1 batch=5 items=7
2026-03-29 10:59:45 [info     ] math_agent_batch_parsed        attempt=1 batch=6 items=1
2026-03-29 10:59:45 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=36
2026-03-29 10:59:45 [info     ] math_agent_generated           items_count=36
2026-03-29 10:59:45 [info     ] reviewer_reviewing             items_count=36 iteration=0
2026-03-29 10:59:51 [info     ] reviewer_completed             iteration=0 passed=36 rejected=0
2026-03-29 10:59:51 [info     ] review_router_deciding         iteration=1 max_iterations=3 pass_rate=1.0 rej

## 5. Multi Game-Type — quiz + flashcard + fill_blank

Each reviewed item is formatted into ALL requested game types.
15 items × 3 types → ~45 total formatted items.

In [6]:
req = make_request(
    'Định luật Newton',
    num_questions=15,
    game_types=['quiz', 'flashcard', 'fill_blank'],
)
res = await run_pipeline(req)
ALL['multi_type'] = summarize(res, 'Multi-type 15Q')

out = res['final_output']
assert out is not None
qn = len(out.content.quiz)
fn = len(out.content.flashcard)
bn = len(out.content.fill_blank)
assert qn >= 10, f'Quiz: {qn}'
assert fn >= 10, f'Flashcard: {fn}'
assert bn >= 10, f'Fill-blank: {bn}'

# Validate flashcard
fc = out.content.flashcard[0]
assert fc.front and fc.back
print(f'\nFlashcard: {fc.front[:80]} → {fc.back[:80]}')

# Validate fill-blank
fb = out.content.fill_blank[0]
assert fb.template and fb.blanks
print(f'Fill-blank: {fb.template[:80]}')
print(f'  Blanks: {[(b.position, b.correct_answer) for b in fb.blanks]}')

print('\n✓ Multi game-type PASSED')

2026-03-29 11:01:50 [info     ] creating_graph                
2026-03-29 11:01:50 [info     ] graph_created                 
2026-03-29 11:01:50 [info     ] graph_compiled_without_checkpointer
2026-03-29 11:01:50 [info     ] supervisor_analyzing           doc_scope=system topic='Định luật Newton' user_id=pipeline_test
2026-03-29 11:01:52 [info     ] supervisor_classified          content_type=math
2026-03-29 11:01:52 [info     ] routing_to_agent               content_type=math route=math_agent
2026-03-29 11:01:52 [info     ] math_agent_starting            doc_scope=system iteration=0 num_questions=15 num_still_needed=15 num_to_generate=18 topic='Định luật Newton' user_id=pipeline_test
2026-03-29 11:01:52 [info     ] retrieving_context             doc_scope=system query='Định luật Newton comprehension level educational content' user_id=pipeline_test
2026-03-29 11:01:52 [debug    ] creating_retriever             doc_scope=system extractive_answers=True filter='user_id: ANY("__system__")

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-29 11:01:54 [debug    ] retrieved_documents            count=30
2026-03-29 11:01:54 [info     ] context_retrieved              chunks=30 sources=30
2026-03-29 11:01:54 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-29 11:02:23 [info     ] math_agent_code_execution_done code_traces_count=6 generation_attempt=1 text_length=9361


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:1076: UserWarning: 
        - 'executable_code': Always present.
        - 'execution_result' & 'image_url': May be absent for some queries.

        Validate before using in production.

  warnings.warn(


2026-03-29 11:02:30 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-29 11:02:39 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=7
2026-03-29 11:02:43 [info     ] math_agent_batch_parsed        attempt=1 batch=3 items=4
2026-03-29 11:02:43 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=18
2026-03-29 11:02:43 [info     ] math_agent_generated           items_count=18
2026-03-29 11:02:43 [info     ] reviewer_reviewing             items_count=18 iteration=0
2026-03-29 11:02:48 [info     ] reviewer_completed             iteration=0 passed=18 rejected=0
2026-03-29 11:02:48 [info     ] review_router_deciding         iteration=1 max_iterations=3 pass_rate=1.0 rejected=0 reviewed=18
2026-03-29 11:02:48 [info     ] review_router_pass             reason=sufficient_items
2026-03-29 11:02:48 [info     ] formatter_trimming             after=15 before=18
2026-03-29 11:02:48 [info     ] formatter_starting             game_types=['qu

## 6. Multi-Chapter Topic — Broad Coverage

A broad topic like "Hàm số" spans chapters:
- Hàm số bậc hai (lớp 10)
- Hàm số mũ & logarit (lớp 12)
- Đạo hàm & ứng dụng (lớp 11-12)

The pipeline should cover sub-topics within the broader topic.

In [7]:
BROAD = [
    ('Hàm số và đồ thị', 20),
    ('Vectơ và hình học không gian', 15),
]

for topic, num_q in BROAD:
    print(f'\n{"="*50}')
    print(f'{topic} ({num_q}Q)')
    print('='*50)

    req = make_request(topic, num_questions=num_q, game_types=['quiz', 'flashcard'])
    res = await run_pipeline(req)
    key = f'broad_{topic[:8]}'
    ALL[key] = summarize(res, topic)

    out = res['final_output']
    if out and out.content.quiz:
        topics = [q.topic for q in out.content.quiz]
        dist = {}
        for t in topics:
            dist[t] = dist.get(t, 0) + 1
        print('Sub-topic distribution:')
        for t, c in sorted(dist.items(), key=lambda x: -x[1]):
            print(f'  {t}: {c}')

print('\n✓ Multi-chapter test completed')


Hàm số và đồ thị (20Q)
2026-03-29 11:05:12 [info     ] creating_graph                
2026-03-29 11:05:12 [info     ] graph_created                 
2026-03-29 11:05:12 [info     ] graph_compiled_without_checkpointer
2026-03-29 11:05:12 [info     ] supervisor_analyzing           doc_scope=system topic='Hàm số và đồ thị' user_id=pipeline_test
2026-03-29 11:05:14 [info     ] supervisor_classified          content_type=math
2026-03-29 11:05:14 [info     ] routing_to_agent               content_type=math route=math_agent
2026-03-29 11:05:14 [info     ] math_agent_starting            doc_scope=system iteration=0 num_questions=20 num_still_needed=20 num_to_generate=24 topic='Hàm số và đồ thị' user_id=pipeline_test
2026-03-29 11:05:14 [info     ] retrieving_context             doc_scope=system query='Hàm số và đồ thị comprehension level educational content' user_id=pipeline_test
2026-03-29 11:05:14 [debug    ] creating_retriever             doc_scope=system extractive_answers=True filter='us

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-29 11:05:16 [debug    ] retrieved_documents            count=30
2026-03-29 11:05:16 [info     ] context_retrieved              chunks=30 sources=30
2026-03-29 11:05:16 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-29 11:06:16 [info     ] math_agent_code_execution_done code_traces_count=6 generation_attempt=1 text_length=13369


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:1076: UserWarning: 
        - 'executable_code': Always present.
        - 'execution_result' & 'image_url': May be absent for some queries.

        Validate before using in production.

  warnings.warn(


2026-03-29 11:06:25 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-29 11:06:37 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=7
2026-03-29 11:06:45 [info     ] math_agent_batch_parsed        attempt=1 batch=3 items=7
2026-03-29 11:06:50 [info     ] math_agent_batch_parsed        attempt=1 batch=4 items=3
2026-03-29 11:06:50 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=24
2026-03-29 11:06:50 [info     ] math_agent_generated           items_count=24
2026-03-29 11:06:50 [info     ] reviewer_reviewing             items_count=24 iteration=0
2026-03-29 11:06:55 [info     ] reviewer_completed             iteration=0 passed=24 rejected=0
2026-03-29 11:06:55 [info     ] review_router_deciding         iteration=1 max_iterations=3 pass_rate=1.0 rejected=0 reviewed=24
2026-03-29 11:06:55 [info     ] review_router_pass             reason=sufficient_items
2026-03-29 11:06:55 [info     ] formatter_trimming             after=20

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-29 11:07:21 [debug    ] retrieved_documents            count=30
2026-03-29 11:07:21 [info     ] context_retrieved              chunks=30 sources=30
2026-03-29 11:07:21 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-29 11:07:48 [info     ] math_agent_code_execution_done code_traces_count=6 generation_attempt=1 text_length=10345


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:1076: UserWarning: 
        - 'executable_code': Always present.
        - 'execution_result' & 'image_url': May be absent for some queries.

        Validate before using in production.

  warnings.warn(


2026-03-29 11:07:53 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-29 11:07:58 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=7
2026-03-29 11:08:07 [info     ] math_agent_batch_parsed        attempt=1 batch=3 items=4
2026-03-29 11:08:07 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=18
2026-03-29 11:08:07 [info     ] math_agent_generated           items_count=18
2026-03-29 11:08:07 [info     ] reviewer_reviewing             items_count=18 iteration=0
2026-03-29 11:08:10 [info     ] reviewer_completed             iteration=0 passed=18 rejected=0
2026-03-29 11:08:10 [info     ] review_router_deciding         iteration=1 max_iterations=3 pass_rate=1.0 rejected=0 reviewed=18
2026-03-29 11:08:10 [info     ] review_router_pass             reason=sufficient_items
2026-03-29 11:08:10 [info     ] formatter_trimming             after=15 before=18
2026-03-29 11:08:10 [info     ] formatter_starting             game_types=['qu

## 7. Difficulty Levels — recall / comprehension / application / high_application

In [8]:
TOPIC = 'Phản ứng oxi hóa khử'

for diff in ['recall', 'comprehension', 'application', 'high_application']:
    print(f'\n--- {diff} (threshold={REVIEW_THRESHOLD_BY_DIFFICULTY[diff]}, overshoot={OVERSHOOT_BY_DIFFICULTY[diff]}x) ---')
    req = make_request(TOPIC, num_questions=10, difficulty=diff, game_types=['quiz'])
    res = await run_pipeline(req)
    ALL[f'diff_{diff}'] = summarize(res, diff)

    out = res['final_output']
    if out and out.content.quiz:
        q = out.content.quiz[0]
        print(f'  Sample: {q.question[:100]}')
        print(f'  Difficulty: {q.difficulty}')
    else:
        print(f'  ⚠️ No quiz items generated for {diff}')

print('\n✓ Difficulty test completed')


--- recall (threshold=0.7, overshoot=1.1x) ---
2026-03-29 11:08:56 [info     ] creating_graph                
2026-03-29 11:08:56 [info     ] graph_created                 
2026-03-29 11:08:56 [info     ] graph_compiled_without_checkpointer
2026-03-29 11:08:56 [info     ] supervisor_analyzing           doc_scope=system topic='Phản ứng oxi hóa khử' user_id=pipeline_test
2026-03-29 11:08:58 [info     ] supervisor_classified          content_type=math
2026-03-29 11:08:58 [info     ] routing_to_agent               content_type=math route=math_agent
2026-03-29 11:08:58 [info     ] math_agent_starting            doc_scope=system iteration=0 num_questions=10 num_still_needed=10 num_to_generate=11 topic='Phản ứng oxi hóa khử' user_id=pipeline_test
2026-03-29 11:08:58 [info     ] retrieving_context             doc_scope=system query='Phản ứng oxi hóa khử recall level educational content' user_id=pipeline_test
2026-03-29 11:08:58 [debug    ] creating_retriever             doc_scope=system extra

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-29 11:09:00 [debug    ] retrieved_documents            count=30
2026-03-29 11:09:00 [info     ] context_retrieved              chunks=30 sources=30
2026-03-29 11:09:00 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-29 11:09:26 [info     ] math_agent_code_execution_done code_traces_count=6 generation_attempt=1 text_length=3605


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:1076: UserWarning: 
        - 'executable_code': Always present.
        - 'execution_result' & 'image_url': May be absent for some queries.

        Validate before using in production.

  warnings.warn(


2026-03-29 11:09:33 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-29 11:09:36 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=4
2026-03-29 11:09:36 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=11
2026-03-29 11:09:36 [info     ] math_agent_generated           items_count=11
2026-03-29 11:09:36 [info     ] reviewer_reviewing             items_count=11 iteration=0
2026-03-29 11:09:41 [info     ] reviewer_completed             iteration=0 passed=11 rejected=0
2026-03-29 11:09:41 [info     ] review_router_deciding         iteration=1 max_iterations=3 pass_rate=1.0 rejected=0 reviewed=11
2026-03-29 11:09:41 [info     ] review_router_pass             reason=sufficient_items
2026-03-29 11:09:41 [info     ] formatter_trimming             after=10 before=11
2026-03-29 11:09:41 [info     ] formatter_starting             game_types=['quiz'] items_count=10 user_id=pipeline_test
2026-03-29 11:09:48 [info     ] formatter_quiz_

...

2026-03-29 11:19:23 [error    ] math_agent_generation_error    error='Failed to parse ContentItemList from completion {"items": [{"question": "Hòa tan hoàn toàn 16,8 gam kim loại Fe vào dung dịch HNO3 loãng, dư. Sau khi phản ứng xảy ra hoàn toàn, thu được V lít (đktc) hỗn hợp khí X gồm NO và N2O có tỉ lệ mol tương ứng là 7:3. Giá trị của V là", "answer": "A", "explanation": "1.  **Tính số mol Fe và số mol electron Fe nhường:**\\n    *   n_Fe = 16,8 / 56 = 0,3 mol.\\n    *   Fe → Fe^(3+) + 3e.\\n    *   n_e_nhuong = 0,3 * 3 = 0,9 mol.\\n\\n2.  **Tính số mol electron do HNO3 nhận:**\\n    *   N^(+5) + 3e → N^(+2) (trong NO)\\n    *   2N^(+5) + 8e → 2N^(+1) (trong N2O)\\n    *   Theo đề, n_NO : n_N2O = 7 : 3. Đặt n_N2O = 3x mol, thì n_NO = 7x mol.\\n    *   Tổng số mol electron nhận = 3 * n_NO + 8 * n"}]}. Got: 1 validation error for ContentItemList\nitems.0.topic\n  Field required [type=missing, input_value={\'question\': \'Hòa tan ho...n = 3 * n_NO + 8 * n\'}, input_type=dict]\n    For 

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-29 11:20:12 [debug    ] retrieved_documents            count=30
2026-03-29 11:20:12 [info     ] context_retrieved              chunks=30 sources=30
2026-03-29 11:20:12 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-29 11:24:18 [info     ] math_agent_code_execution_done code_traces_count=40 generation_attempt=1 text_length=85921


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:1076: UserWarning: 
        - 'executable_code': Always present.
        - 'execution_result' & 'image_url': May be absent for some queries.

        Validate before using in production.

  warnings.warn(


2026-03-29 11:24:40 [warning  ] rate_limiter_provider_error    consecutive_failures=1 error='Failed to parse ContentItemList from completion {"items": [{"question": "Cho 18,5 gam hỗn hợp X gồm Fe và Fe₃O₄ tác dụng hoàn toàn với dung dịch HNO₃ loãng, đun nóng và khuấy đều. Sau khi phản ứng xảy'
2026-03-29 11:24:40 [error    ] math_agent_generation_error    error='Failed to parse ContentItemList from completion {"items": [{"question": "Cho 18,5 gam hỗn hợp X gồm Fe và Fe₃O₄ tác dụng hoàn toàn với dung dịch HNO₃ loãng, đun nóng và khuấy đều. Sau khi phản ứng xảy ra hoàn toàn, thu được 2,24 lít khí NO (sản phẩm khử duy nhất ở đktc) và dung dịch Y chỉ chứa Fe(NO₃)₃. Cô cạn dung dịch Y thu được m gam muối khan. Giá trị của m là bao nhiêu?", "answer": "m = 63,2225 gam.", "explanation": "Gọi số mol Fe là x và số mol Fe₃O₄ là y.\\nKhối lượng hỗn hợp X: 56x + 232y = 18,5 (1)\\nSố mol NO = 2,24 / 22,4 = 0,1 mol.\\n\\nQuá trình oxi hóa:\\nFe⁰ → Fe³⁺ + 3e (x mol Fe nhường 3x mol e)\\nFe₃O₄ (Fe⁺⁸⁄₃)

## 8. STEM Subject Coverage (20 questions each, all game types)

Real-world usage: generate content for an entire lesson across subjects.

In [9]:
SUBJECTS = [
    ('Toán', 'Giải hệ phương trình bậc nhất hai ẩn', 20),
    ('Lý', 'Chuyển động thẳng đều và biến đổi đều', 20),
    ('Hóa', 'Cấu hình electron và bảng tuần hoàn', 20),
]

for subj, topic, nq in SUBJECTS:
    print(f'\n{"="*50}')
    print(f'[{subj}] {topic} ({nq}Q × 3 types)')
    print('='*50)

    req = make_request(topic, num_questions=nq, game_types=['quiz', 'flashcard', 'fill_blank'])
    res = await run_pipeline(req)
    ALL[f'subj_{subj}'] = summarize(res, subj)

    out = res['final_output']
    if out:
        traces = sum(1 for q in out.content.quiz if q.computation_trace)
        print(f'  Computation traces: {traces}/{len(out.content.quiz)}')

print('\n✓ Subject coverage test completed')


[Toán] Giải hệ phương trình bậc nhất hai ẩn (20Q × 3 types)
2026-03-29 11:26:12 [info     ] creating_graph                
2026-03-29 11:26:12 [info     ] graph_created                 
2026-03-29 11:26:12 [info     ] graph_compiled_without_checkpointer
2026-03-29 11:26:12 [info     ] supervisor_analyzing           doc_scope=system topic='Giải hệ phương trình bậc nhất hai ẩn' user_id=pipeline_test
2026-03-29 11:26:14 [info     ] supervisor_classified          content_type=math
2026-03-29 11:26:14 [info     ] routing_to_agent               content_type=math route=math_agent
2026-03-29 11:26:14 [info     ] math_agent_starting            doc_scope=system iteration=0 num_questions=20 num_still_needed=20 num_to_generate=24 topic='Giải hệ phương trình bậc nhất hai ẩn' user_id=pipeline_test
2026-03-29 11:26:14 [info     ] retrieving_context             doc_scope=system query='Giải hệ phương trình bậc nhất hai ẩn comprehension level educational content' user_id=pipeline_test
2026-03-29 11:26:

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-29 11:26:16 [debug    ] retrieved_documents            count=30
2026-03-29 11:26:16 [info     ] context_retrieved              chunks=30 sources=30
2026-03-29 11:26:16 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-29 11:27:25 [info     ] math_agent_code_execution_done code_traces_count=10 generation_attempt=1 text_length=14372


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:1076: UserWarning: 
        - 'executable_code': Always present.
        - 'execution_result' & 'image_url': May be absent for some queries.

        Validate before using in production.

  warnings.warn(


2026-03-29 11:27:29 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-29 11:27:39 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=7
2026-03-29 11:27:51 [info     ] math_agent_batch_parsed        attempt=1 batch=3 items=7
2026-03-29 11:27:56 [info     ] math_agent_batch_parsed        attempt=1 batch=4 items=3
2026-03-29 11:27:56 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=24
2026-03-29 11:27:56 [info     ] math_agent_generated           items_count=24
2026-03-29 11:27:56 [info     ] reviewer_reviewing             items_count=24 iteration=0
2026-03-29 11:27:59 [info     ] reviewer_completed             iteration=0 passed=24 rejected=0
2026-03-29 11:27:59 [info     ] review_router_deciding         iteration=1 max_iterations=3 pass_rate=1.0 rejected=0 reviewed=24
2026-03-29 11:27:59 [info     ] review_router_pass             reason=sufficient_items
2026-03-29 11:27:59 [info     ] formatter_trimming             after=20

...

2026-03-29 11:30:21 [debug    ] retrieved_documents            count=30
2026-03-29 11:30:21 [info     ] context_retrieved              chunks=30 sources=30
2026-03-29 11:30:21 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-29 11:30:49 [info     ] math_agent_code_execution_done code_traces_count=0 generation_attempt=1 text_length=1279
2026-03-29 11:31:00 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-29 11:31:15 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=7
2026-03-29 11:31:29 [info     ] math_agent_batch_parsed        attempt=1 batch=3 items=7
2026-03-29 11:31:32 [info     ] math_agent_batch_parsed        attempt=1 batch=4 items=3
2026-03-29 11:31:32 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=24
2026-03-29 11:31:32 [info     ] math_agent_generated           items_count=24
2026-03-29 11:31:32 [info     ] reviewer_reviewing             items_count=24 iteration=0
2026-03-29 11:31:37 [info

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-29 11:31:41 [debug    ] retrieved_documents            count=28
2026-03-29 11:31:41 [info     ] context_retrieved              chunks=28 sources=28
2026-03-29 11:31:41 [info     ] math_agent_context_retrieved   context_chunks=28
2026-03-29 11:32:23 [info     ] math_agent_code_execution_done code_traces_count=2 generation_attempt=1 text_length=13568


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:1076: UserWarning: 
        - 'executable_code': Always present.
        - 'execution_result' & 'image_url': May be absent for some queries.

        Validate before using in production.

  warnings.warn(


2026-03-29 11:32:41 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-29 11:32:55 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=5
2026-03-29 11:32:55 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=12
2026-03-29 11:32:55 [info     ] math_agent_generated           items_count=12
2026-03-29 11:32:55 [info     ] reviewer_reviewing             items_count=12 iteration=1
2026-03-29 11:32:58 [info     ] reviewer_completed             iteration=1 passed=12 rejected=0
2026-03-29 11:32:58 [info     ] review_router_deciding         iteration=2 max_iterations=3 pass_rate=0.6111111111111112 rejected=14 reviewed=22
2026-03-29 11:32:58 [info     ] review_router_pass             reason=sufficient_items
2026-03-29 11:32:58 [info     ] formatter_trimming             after=20 before=22
2026-03-29 11:32:58 [info     ] formatter_starting             game_types=['quiz', 'flashcard', 'fill_blank'] items_count=20 user_id=pipeline_test
2026

## 9. Quality Metrics Summary

In [13]:
print('=' * 90)
print('PIPELINE TEST SUMMARY (P0–P4 Fixes Applied)')
print('=' * 90)
print(f'{"Test":<22} {"Total":>6} {"Quiz":>5} {"FC":>4} {"FB":>4} {"Gen":>5} {"Pass%":>6} {"Iter":>4} {"Time":>7}')
print('-' * 90)

total_items = 0
total_time = 0.0
pass_rates = []

# Old baseline for comparison
OLD_BASELINE = {
    'smoke_5q': 100.0, 'medium_15q': 57.7, 'large_30q': 83.3,
    'multi_type': 100.0, 'diff_recall': 100.0, 'diff_comprehension': 100.0,
    'diff_application': 90.9, 'diff_high_application': 0.0,
}

for name, s in ALL.items():
    if s is None:
        print(f'{name:<22} FAILED')
        continue
    pr = s['passed'] / s['generated'] * 100 if s['generated'] > 0 else 0
    pass_rates.append(pr)
    total_items += s['total']
    total_time += s['elapsed']
    it = s.get('iterations', '?')
    old = OLD_BASELINE.get(name)
    delta = f' ({"+" if pr >= old else ""}{pr - old:+.1f}pp)' if old is not None else ''
    print(f'{name:<22} {s["total"]:>6} {s["quiz"]:>5} {s["fc"]:>4} {s["fb"]:>4} {s["generated"]:>5} {pr:>5.1f}%{delta:<12} {it:>4} {s["elapsed"]:>6.1f}s')

print('-' * 90)
avg_pr = sum(pass_rates) / len(pass_rates) if pass_rates else 0
print(f'{"TOTAL":<22} {total_items:>6} {"":>5} {"":>4} {"":>4} {"":>5} {avg_pr:>5.1f}%{"":>12} {"":>4} {total_time:>6.1f}s')

print(f'\nTests: {len(ALL)} | Items: {total_items} | Avg pass rate: {avg_pr:.1f}% (old: 84.6%) | Time: {total_time:.0f}s ({total_time/60:.1f}min)')

failed = [k for k, v in ALL.items() if v is None]
if failed:
    print(f'❌ Failed: {failed}')
else:
    print('✅ All pipeline tests passed')

PIPELINE TEST SUMMARY (P0–P4 Fixes Applied)
Test                    Total  Quiz   FC   FB   Gen  Pass% Iter    Time
------------------------------------------------------------------------------------------
smoke_5q                    5     5    0    0     5 100.0% (++0.0pp)      1   60.8s
medium_15q                 15    15    0    0    15 100.0% (++42.3pp)     1  228.1s
large_30q                  30    30    0    0    30 100.0% (++16.7pp)     1  210.7s
multi_type                 45    15   15   15    15 100.0% (++0.0pp)      1   80.2s
broad_Hàm số v             40    20   20    0    20 100.0%                1  125.3s
broad_Vectơ và             30    15   15    0    15 100.0%                1   76.6s
diff_recall                10    10    0    0    10 100.0% (++0.0pp)      1   54.8s
diff_comprehension         10    10    0    0    10 100.0% (++0.0pp)      1   93.9s
diff_application           10    10    0    0    10 100.0% (++9.1pp)      1  159.0s
diff_high_application       3     3  

In [15]:
# ── P0–P4 Fix Verification ──
print('='*70)
print('FIX VERIFICATION')
print('='*70)

# P0: Feedback loop — check if retry iterations were used
retry_tests = {k: v for k, v in ALL.items() if v and v.get('iterations', 0) > 1}
print(f'\nP0 Feedback Loop: {len(retry_tests)} test(s) used retry iterations')
for name, s in retry_tests.items():
    print(f'  {name}: {s["iterations"]} iterations')

# P1 + P3: high_application should now produce items (was 0% before)
ha = ALL.get('diff_high_application')
ha_rerun = ALL.get('diff_high_application_rerun')
if ha:
    ha_pr = ha['passed'] / ha['generated'] * 100 if ha['generated'] > 0 else 0
    old_pr = 0.0
    print(f'\nP1+P3 high_application (baseline run): {ha_pr:.1f}% pass rate (old: {old_pr:.1f}%)')
    print(f'  gen={ha["generated"]}, passed={ha["passed"]}, rejected={ha["rejected"]}')
    print(f'  quiz={ha["quiz"]}, fc={ha["fc"]}, fb={ha["fb"]}')
    if ha_pr > old_pr:
        print(f'  ✅ IMPROVED by {ha_pr - old_pr:+.1f}pp')
    else:
        print(f'  ⚠️ No improvement')

if ha_rerun:
    ha2_pr = ha_rerun['passed'] / ha_rerun['generated'] * 100 if ha_rerun['generated'] > 0 else 0
    print(f'\nFocused rerun high_application: {ha2_pr:.1f}% pass rate')
    print(f'  gen={ha_rerun["generated"]}, passed={ha_rerun["passed"]}, rejected={ha_rerun["rejected"]}')
    if ha:
        delta = ha2_pr - (ha['passed'] / ha['generated'] * 100 if ha['generated'] > 0 else 0)
        print(f'  Δ vs baseline run: {delta:+.1f}pp')

# P2: Adaptive overshoot — verify diff tests used different generation counts
print(f'\nP2 Adaptive Overshoot:')
for diff in ['recall', 'comprehension', 'application', 'high_application']:
    d = ALL.get(f'diff_{diff}')
    if d:
        overshoot = OVERSHOOT_BY_DIFFICULTY[diff]
        expected_gen = int(10 * overshoot) + 1  # ceil
        print(f'  {diff}: generated={d["generated"]} (expected ~{int(10 * overshoot)}-{expected_gen} with {overshoot}x overshoot)')

# Overall comparison
valid = [v for v in ALL.values() if v and v['generated'] > 0]
overall = sum(v['passed']/v['generated']*100 for v in valid)/len(valid) if valid else 0
print(f'\nOverall: avg pass rate = {overall:.1f}% (old: 84.6%)')

if ha_rerun and (ha_rerun['passed'] / ha_rerun['generated'] * 100 if ha_rerun['generated'] > 0 else 0) < 30:
    print('⚠️ high_application vẫn là suspicious chính: cần tối ưu tiếp prompt + exemplar theo domain Hóa.')

FIX VERIFICATION

P0 Feedback Loop: 3 test(s) used retry iterations
  diff_high_application: 3 iterations
  subj_Hóa: 2 iterations
  diff_high_application_rerun: 3 iterations

P1+P3 high_application (baseline run): 6.2% pass rate (old: 0.0%)
  gen=48, passed=3, rejected=45
  quiz=3, fc=0, fb=0
  ✅ IMPROVED by +6.2pp

Focused rerun high_application: 10.0% pass rate
  gen=20, passed=2, rejected=18
  Δ vs baseline run: +3.8pp

P2 Adaptive Overshoot:
  recall: generated=10 (expected ~11-12 with 1.1x overshoot)
  comprehension: generated=10 (expected ~12-13 with 1.2x overshoot)
  application: generated=10 (expected ~14-15 with 1.4x overshoot)
  high_application: generated=48 (expected ~20-21 with 2.0x overshoot)

Overall: avg pass rate = 83.9% (old: 84.6%)
⚠️ high_application vẫn là suspicious chính: cần tối ưu tiếp prompt + exemplar theo domain Hóa.


In [12]:
# 10. Suspicious Auto-RCA + 11. Focus rerun for diff_high_application

# Reload key modules so this rerun reflects newest code changes in source files.
import importlib
import src.config.constants as _constants_mod
import src.graph.nodes.math_agent as _math_agent_mod
import src.graph.nodes.reviewer as _reviewer_mod
import src.graph.builder as _builder_mod

importlib.reload(_constants_mod)
importlib.reload(_math_agent_mod)
importlib.reload(_reviewer_mod)
importlib.reload(_builder_mod)
from src.graph.builder import compile_graph as _compile_graph_new
globals()['compile_graph'] = _compile_graph_new

def _pass_rate(s: dict | None) -> float:
    if not s or s.get('generated', 0) <= 0:
        return 0.0
    return s['passed'] / s['generated'] * 100.0

def _build_suspicious_report(all_results: dict[str, dict]) -> list[dict]:
    suspicious = []
    for name, s in all_results.items():
        if s is None:
            suspicious.append({
                'name': name,
                'severity': 'critical',
                'pass_rate': 0.0,
                'causes': ['Không có output hợp lệ từ pipeline.'],
                'solutions': [
                    'Kiểm tra log lỗi node gây fail (math_agent/reviewer/formatter).',
                    'Giảm batch size hoặc test scope để cô lập lỗi.',
                ],
            })
            continue

        pr = _pass_rate(s)
        iterations = s.get('iterations', 0)
        generated = s.get('generated', 0)
        total = s.get('total', 0)

        is_suspicious = (
            pr < 70.0
            or iterations > 1
            or (total > 0 and generated > total * 2)
        )
        if not is_suspicious:
            continue

        causes = []
        solutions = []
        severity = 'warning'

        if pr < 40.0:
            severity = 'critical'
        elif pr < 70.0:
            severity = 'high'

        if 'diff_high_application' in name and pr < 70.0:
            causes.extend([
                'Mức high_application vẫn khó, nhiều câu bị reviewer loại.',
                'Có thể lệch domain exemplar so với topic test.',
                'Chất lượng lập luận chưa đạt mức non-routine reasoning ổn định.',
            ])
            solutions.extend([
                'Tăng exemplar theo đúng domain/topic (đặc biệt Hóa cho topic hiện tại).',
                'Giảm ngưỡng pass cho high_application ở mức an toàn (ví dụ 0.50).',
                'Rerun focused test và đọc feedback của rejected_items để tinh chỉnh prompt.',
            ])

        if iterations > 1:
            causes.append('Feedback loop đã kích hoạt, chứng tỏ vòng sinh đầu chưa đủ chất lượng.')
            solutions.append('Giữ retry nhưng tối ưu prompt từ feedback gần nhất theo đúng difficulty.')

        if total > 0 and generated > total * 2:
            causes.append('Số lượng generated cao bất thường so với output cuối -> hiệu suất thấp.')
            solutions.append('Điều chỉnh overshoot hoặc tăng chất lượng generation để giảm hao hụt qua review.')

        if not causes:
            causes.append('Pass rate thấp hơn ngưỡng kỳ vọng.')
            solutions.append('Kiểm tra lại prompt + threshold + parse reliability cho test này.')

        suspicious.append({
            'name': name,
            'severity': severity,
            'pass_rate': pr,
            'causes': causes,
            'solutions': solutions,
            'iterations': iterations,
            'generated': generated,
            'total': total,
        })

    suspicious.sort(key=lambda x: (x['pass_rate'], -x['iterations']))
    return suspicious

print('=' * 90)
print('SUSPICIOUS AUTO-RCA REPORT')
print('=' * 90)
sus = _build_suspicious_report(ALL)
if not sus:
    print('✅ Không có suspicious theo rule hiện tại.')
else:
    for i, item in enumerate(sus, 1):
        print(f"\n[{i}] {item['name']} | severity={item['severity']} | pass={item['pass_rate']:.1f}% | iter={item['iterations']}")
        print(f"    generated={item['generated']} total={item['total']}")
        print('    Causes:')
        for c in item['causes'][:3]:
            print(f'      - {c}')
        print('    Solutions:')
        for s in item['solutions'][:3]:
            print(f'      - {s}')

# Focus rerun for diff_high_application
FOCUS_RERUN = True
if FOCUS_RERUN:
    print('\n' + '=' * 90)
    print('FOCUS RERUN: diff_high_application')
    print('=' * 90)
    req = make_request(
        'Phản ứng oxi hóa khử',
        num_questions=10,
        difficulty='high_application',
        game_types=['quiz'],
    )
    res = await run_pipeline(req)
    ALL['diff_high_application_rerun'] = summarize(res, 'high_application_rerun')

    out = res.get('final_output')
    if out and out.content.quiz:
        print(f"Sample question: {out.content.quiz[0].question[:160]}")
        print(f"Difficulty label: {out.content.quiz[0].difficulty}")
    else:
        print('⚠️ Focus rerun returned no quiz items.')

SUSPICIOUS AUTO-RCA REPORT

[1] diff_high_application | severity=critical | pass=6.2% | iter=3
    generated=48 total=3
    Causes:
      - Mức high_application vẫn khó, nhiều câu bị reviewer loại.
      - Có thể lệch domain exemplar so với topic test.
      - Chất lượng lập luận chưa đạt mức non-routine reasoning ổn định.
    Solutions:
      - Tăng exemplar theo đúng domain/topic (đặc biệt Hóa cho topic hiện tại).
      - Giảm ngưỡng pass cho high_application ở mức an toàn (ví dụ 0.50).
      - Rerun focused test và đọc feedback của rejected_items để tinh chỉnh prompt.

[2] subj_Hóa | severity=high | pass=58.8% | iter=2
    generated=34 total=60
    Causes:
      - Feedback loop đã kích hoạt, chứng tỏ vòng sinh đầu chưa đủ chất lượng.
    Solutions:
      - Giữ retry nhưng tối ưu prompt từ feedback gần nhất theo đúng difficulty.

FOCUS RERUN: diff_high_application
2026-03-29 12:08:11 [info     ] creating_graph                
2026-03-29 12:08:11 [info     ] graph_created             

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-29 12:08:16 [debug    ] retrieved_documents            count=30
2026-03-29 12:08:16 [info     ] context_retrieved              chunks=30 sources=30
2026-03-29 12:08:16 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-29 12:08:48 [info     ] math_agent_code_execution_done code_traces_count=0 generation_attempt=1 text_length=14080
2026-03-29 12:09:08 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-29 12:09:17 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=7
2026-03-29 12:09:31 [info     ] math_agent_batch_parsed        attempt=1 batch=3 items=6
2026-03-29 12:09:31 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=20
2026-03-29 12:09:31 [info     ] math_agent_generated           items_count=20
2026-03-29 12:09:31 [info     ] reviewer_reviewing             items_count=20 iteration=0
2026-03-29 12:09:36 [info     ] reviewer_completed             iteration=0 passed=2 rejected=18
2026-03-29 12:09:

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-29 12:09:40 [debug    ] retrieved_documents            count=30
2026-03-29 12:09:40 [info     ] context_retrieved              chunks=30 sources=30
2026-03-29 12:09:40 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-29 12:10:37 [info     ] math_agent_code_execution_done code_traces_count=2 generation_attempt=1 text_length=19664


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:1076: UserWarning: 
        - 'executable_code': Always present.
        - 'execution_result' & 'image_url': May be absent for some queries.

        Validate before using in production.

  warnings.warn(


2026-03-29 12:11:04 [error    ] math_agent_generation_error    error='Failed to parse ContentItemList from completion {"items": [{"question": "Cho 15,8 gam KMnO₄ tác dụng hết với dung dịch HCl đặc, dư. Giả sử hiệu suất phản ứng là 100%. Nếu toàn bộ khí Cl₂ sinh ra được hấp thụ vào 200 ml dung dịch NaOH 2M ở nhiệt độ thường, sau đó thêm dung dịch AgNO₃ dư vào dung dịch sau phản ứng thì thu được bao nhiêu gam kết tủa?", "answer": "28,7 gam.", "explanation": "1.  **Phản ứng giữa KMnO₄ và HCl:**\\n    2KMnO₄ + 16HCl → 2KCl + 2MnCl₂ + 5Cl₂ + 8H₂O\\n    Số mol KMnO₄ = 15,8 / 158 = 0,1 mol.\\n    Theo phương trình, số mol Cl₂ = (0,1 * 5) / 2 = 0,25 mol.\\n\\n2.  **Phản ứng giữa Cl₂ và NaOH (nhiệt độ thường):**\\n    Cl₂ + 2NaOH → NaCl + NaClO + H₂O\\n    Số mol NaOH = 0,2 L * 2 M = 0,4 mol.\\n    Tỉ lệ mol Cl₂ : NaOH = 0,25 : 0,4 = 1 : 1,6.\\n    Vì tỉ lệ này nằm giữa 1:1 và 1:2, Cl₂ phản ứng hết, NaOH dư.\\n    Sau phản ứng, dung dịch chứa NaCl, NaClO và NaOH dư.\\n    Số mol NaCl = số mol C

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-29 12:16:56 [debug    ] retrieved_documents            count=30
2026-03-29 12:16:56 [info     ] context_retrieved              chunks=30 sources=30
2026-03-29 12:16:56 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-29 12:17:30 [info     ] math_agent_code_execution_done code_traces_count=0 generation_attempt=1 text_length=899
2026-03-29 12:17:45 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-29 12:18:16 [error    ] math_agent_generation_error    error='Failed to parse ContentItemList from completion {"items": [{"question": "Cho 23,2 gam hỗn hợp X gồm Fe, FeO, Fe2O3 và Fe3O4 tác dụng vừa đủ với 800 ml dung dịch HNO3 1M, sau phản ứng thu được dung dịch Y và 2,24 lít (đktc) khí NO (sản phẩm khử duy nhất của N+5). Cô cạn cẩn thận dung dịch Y thu được m gam muối khan. Tính giá trị của m.", "answer": "m = 72,6 gam.", "explanation": "Coi hỗn hợp X gồm Fe và O. \\n$n_{NO} = \\\\frac{2,24}{22,4} = 0,1 \\\\text{ mol}$\\nÁp dụng định luật

## Round 2: Model Upgrade + Exemplar Enrichment + Feedback Classification
**Changes:**
- Generation model: `gemini-2.5-flash` → `gemini-3-flash-preview` (Pro-level reasoning)
- Review model: `gemini-3.1-flash-lite-preview` → `gemini-3-flash-preview`
- Enriched high_application exemplars: +2 chemistry, +1 physics, +1 biology
- Anti-pattern guidance added to high_application prompt
- Reject feedback now classified by error category with targeted fix guidance

In [16]:
# ── Round 2: Reload modules with new code + clear LLM cache ──
import importlib
import src.config.settings as _settings_mod
import src.config.constants as _constants_mod
import src.services.llm as _llm_mod
import src.graph.nodes.math_agent as _math_agent_mod
import src.graph.nodes.reviewer as _reviewer_mod
import src.graph.builder as _builder_mod

for mod in [_settings_mod, _constants_mod, _llm_mod, _math_agent_mod, _reviewer_mod, _builder_mod]:
    importlib.reload(mod)

# Clear LLM cache — critical after model name change
_llm_mod.get_generation_llm.cache_clear()
_llm_mod.get_review_llm.cache_clear()

# Clear settings cache so new model names are picked up
_settings_mod.get_settings.cache_clear() if hasattr(_settings_mod, 'get_settings') else None
from src.config import get_settings as _gs
_gs.cache_clear()

from src.graph.builder import compile_graph as _compile_graph_r2
globals()['compile_graph'] = _compile_graph_r2

# Verify model upgrade
s2 = _gs()
print(f"Generation model: {s2.generation_model}")
print(f"Review model:     {s2.review_model}")
print(f"Gen location:     {s2.generation_model_location or s2.gcp_location}")
print(f"Review location:  {s2.review_model_location or s2.gcp_location}")
assert 'gemini-3' in s2.generation_model, f"Expected gemini-3 model, got {s2.generation_model}"
print("✅ Model upgrade confirmed")

Generation model: gemini-3-flash-preview
Review model:     gemini-3-flash-preview
Gen location:     asia-southeast1
Review location:  asia-southeast1
✅ Model upgrade confirmed


In [29]:
# ── Round 2: Focus rerun for high_application with upgraded model ──
import asyncio, time

req_r2 = make_request(
    'Phản ứng oxi hóa khử',
    num_questions=10,
    difficulty='high_application',
    game_types=['quiz'],
)

t0 = time.time()
res_r2 = await run_pipeline(req_r2)
t1 = time.time()

ALL['r2_high_application'] = summarize(res_r2, 'r2_high_application')
s_r2 = ALL['r2_high_application']
pr_r2 = s_r2['passed'] / s_r2['generated'] * 100 if s_r2['generated'] > 0 else 0

print(f"Round 2 high_application: {pr_r2:.1f}% pass rate")
print(f"  generated={s_r2['generated']}, passed={s_r2['passed']}, rejected={s_r2['rejected']}")
print(f"  quiz={s_r2.get('quiz',0)}, iterations={s_r2.get('iterations',0)}")
print(f"  time={t1-t0:.1f}s")

# Compare with Round 1
r1 = ALL.get('diff_high_application_rerun') or ALL.get('diff_high_application')
if r1:
    pr_r1 = r1['passed'] / r1['generated'] * 100 if r1['generated'] > 0 else 0
    delta = pr_r2 - pr_r1
    print(f"\nΔ vs Round 1: {delta:+.1f}pp ({pr_r1:.1f}% → {pr_r2:.1f}%)")
    if delta > 0:
        print(f"  ✅ IMPROVED")
    else:
        print(f"  ⚠️ No improvement")

# Show sample question
out_r2 = res_r2.get('final_output')
if out_r2 and out_r2.content.quiz:
    q0 = out_r2.content.quiz[0]
    print(f"\nSample Q: {q0.question[:200]}")
    print(f"Sample A: option {q0.correct_answer_index}")
else:
    print("⚠️ No quiz items in output")

2026-03-29 15:33:18 [info     ] creating_graph                
2026-03-29 15:33:18 [info     ] graph_created                 
2026-03-29 15:33:18 [info     ] graph_compiled_without_checkpointer
2026-03-29 15:33:18 [info     ] supervisor_analyzing           doc_scope=system topic='Phản ứng oxi hóa khử' user_id=pipeline_test
2026-03-29 15:33:20 [info     ] supervisor_classified          content_type=math
2026-03-29 15:33:20 [info     ] routing_to_agent               content_type=math route=math_agent
2026-03-29 15:33:20 [info     ] math_agent_starting            doc_scope=system iteration=0 num_questions=10 num_still_needed=10 num_to_generate=20 topic='Phản ứng oxi hóa khử' user_id=pipeline_test
2026-03-29 15:33:20 [info     ] retrieving_context             doc_scope=system query='Phản ứng oxi hóa khử high_application level educational content' user_id=pipeline_test
2026-03-29 15:33:20 [debug    ] creating_retriever             doc_scope=system extractive_answers=True filter='user_id: AN

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-29 15:33:23 [debug    ] retrieved_documents            count=30
2026-03-29 15:33:23 [info     ] context_retrieved              chunks=30 sources=30
2026-03-29 15:33:23 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-29 15:34:17 [info     ] math_agent_code_execution_done code_traces_count=2 generation_attempt=1 text_length=10669


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:1076: UserWarning: 
        - 'executable_code': Always present.
        - 'execution_result' & 'image_url': May be absent for some queries.

        Validate before using in production.

  warnings.warn(


2026-03-29 15:34:47 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-29 15:35:11 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=7
2026-03-29 15:35:35 [info     ] math_agent_batch_parsed        attempt=1 batch=3 items=6
2026-03-29 15:35:35 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=20
2026-03-29 15:35:35 [info     ] math_agent_generated           items_count=20
2026-03-29 15:35:35 [info     ] reviewer_reviewing             items_count=20 iteration=0
2026-03-29 15:35:57 [info     ] reviewer_completed             iteration=0 passed=1 rejected=1
2026-03-29 15:35:57 [info     ] review_router_deciding         iteration=1 max_iterations=3 pass_rate=0.5 rejected=1 reviewed=1
2026-03-29 15:35:57 [info     ] review_router_fail             have=1 need=7 reason=insufficient_items
2026-03-29 15:35:57 [info     ] supervisor_analyzing           doc_scope=system topic='Phản ứng oxi hóa khử' user_id=pipeline_test
2026-03-29 15:3

...

2026-03-29 15:37:35 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-29 15:38:06 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=7
2026-03-29 15:38:25 [info     ] math_agent_batch_parsed        attempt=1 batch=3 items=4
2026-03-29 15:38:25 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=18
2026-03-29 15:38:25 [info     ] math_agent_generated           items_count=18
2026-03-29 15:38:25 [info     ] reviewer_reviewing             items_count=18 iteration=1
2026-03-29 15:38:49 [info     ] reviewer_completed             iteration=1 passed=3 rejected=0
2026-03-29 15:38:49 [info     ] review_router_deciding         iteration=2 max_iterations=3 pass_rate=0.8 rejected=1 reviewed=4
2026-03-29 15:38:49 [info     ] review_router_fail             have=4 need=7 reason=insufficient_items
2026-03-29 15:38:50 [info     ] supervisor_analyzing           doc_scope=system topic='Phản ứng oxi hóa khử' user_id=pipeline_test
2026-03-29 15:3

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-29 15:38:54 [debug    ] retrieved_documents            count=30
2026-03-29 15:38:54 [info     ] context_retrieved              chunks=30 sources=30
2026-03-29 15:38:54 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-29 15:40:39 [info     ] math_agent_code_execution_done code_traces_count=4 generation_attempt=1 text_length=8547


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:1076: UserWarning: 
        - 'executable_code': Always present.
        - 'execution_result' & 'image_url': May be absent for some queries.

        Validate before using in production.

  warnings.warn(


2026-03-29 15:41:28 [error    ] math_agent_generation_error    error='Failed to parse ContentItemList from completion {"items": [{"question": "Câu 1. (Xác định thành phần hỗn hợp)\\nHỗn hợp X gồm Cu và Fe3O4. Cho 17,28 gam X vào dung dịch HNO3 loãng dư, sau khi phản ứng xảy ra hoàn toàn thu được 0,448 lít khí NO (sản phẩm khử duy nhất, đktc). Nếu hòa tan hoàn toàn 17,28 gam X vào dung dịch HCl dư, khối lượng chất rắn không tan thu được là:\\nA. 1,28 gam.\\nB. 6,40 gam.\\nC. 3,20 gam.\\nD. 0,64 gam.", "answer": "C. 3,20 gam.", "explanation": "nNO = 0,02 mol. Gọi nCu = a, nFe3O4 = b. Ta có hệ phương trình: 64a + 232b = 17,28 và bảo toàn electron: 2a + b = 3 * 0,02 = 0,06. Giải hệ thu được a = 0,08 và b = 0,052. Khi cho X vào HCl dư: Fe3O4 + 8HCl -> FeCl2 + 2FeCl3 + 4H2O. nFeCl3 = 2 * nFe3O4"}]}. Got: 1 validation error for ContentItemList\nitems.0.topic\n  Field required [type=missing, input_value={\'question\': \'Câu 1. (X...O. nFeCl3 = 2 * nFe3O4\'}, input_type=dict]\n    For further i

In [30]:
# ── Read Round 2 results (already in kernel from previous execution) ──
s_r2 = ALL.get('r2_high_application')
if s_r2:
    pr_r2 = s_r2['passed'] / s_r2['generated'] * 100 if s_r2['generated'] > 0 else 0
    print(f"Round 2 high_application: {pr_r2:.1f}% pass rate")
    print(f"  generated={s_r2['generated']}, passed={s_r2['passed']}, rejected={s_r2['rejected']}")
    print(f"  quiz={s_r2.get('quiz',0)}, iterations={s_r2.get('iterations',0)}")
    
    # Compare with historical baseline
    print(f"\nHistorical baseline: ~10.0%")
    delta = pr_r2 - 10.0
    print(f"Δ vs historical: {delta:+.1f}pp (10.0% → {pr_r2:.1f}%)")

    # Show sample question
    out_r2 = res_r2.get('final_output')
    if out_r2 and out_r2.content.quiz:
        q0 = out_r2.content.quiz[0]
        print(f"\nSample Q: {q0.question[:200]}")
        print(f"Sample A: option {q0.correct_answer_index}")
    else:
        print("⚠️ No quiz items in output")
else:
    print("⚠️ No r2_high_application data found")

Round 2 high_application: 71.4% pass rate
  generated=7, passed=5, rejected=2
  quiz=3, iterations=3

Historical baseline: ~10.0%
Δ vs historical: +61.4pp (10.0% → 71.4%)

Sample Q: Khi cho 15,8 gam $KMnO_4$ tác dụng hoàn toàn với dung dịch $HCl$ đặc, dư, thu được $V$ lít khí $Cl_2$ (đkc). Giá trị của $V$ và số mol $HCl$ đóng vai trò là chất khử trong phản ứng này lần lượt là:
Sample A: option 0


In [26]:
# ── Round 2: Detailed analysis ──
out_r2 = res_r2.get('final_output')
state_r2 = res_r2

# Final output items
if out_r2:
    c = out_r2.content
    print(f"Final output: quiz={len(c.quiz)}, flashcard={len(c.flashcard)}, fill_blank={len(c.fill_blank)}")
    m = out_r2.metadata
    print(f"Metadata: generated={m.total_generated}, passed={m.total_passed_review}, rejected={m.total_rejected}")

# Reviewed items (all that passed review across iterations)
reviewed = state_r2.get('reviewed_items', [])
print(f"\nReviewed items in state: {len(reviewed)}")
for i, item in enumerate(reviewed[:3]):
    q = item.get('question', item.get('content', ''))[:100]
    score = item.get('review_score', 'N/A')
    print(f"  [{i}] score={score}: {q}")

# Rejected items
rejected = state_r2.get('rejected_items', [])
print(f"\nRejected items: {len(rejected)}")
for i, item in enumerate(rejected):
    q = item.get('question', item.get('content', ''))[:100]
    reason = item.get('rejection_reason', item.get('review_feedback', 'N/A'))[:200]
    print(f"  [{i}] {q}")
    print(f"      reason: {reason}")

# Errors
errors = state_r2.get('errors', [])
if errors:
    print(f"\nErrors: {errors}")

# Iteration details
print(f"\nIterations: {state_r2.get('iteration_count', 0)}")

Final output: quiz=4, flashcard=0, fill_blank=0
Metadata: generated=13, passed=10, rejected=3

Reviewed items in state: 18
  [0] score=1.0: (Quy đổi hỗn hợp) Hòa tan hoàn toàn 11,36 gam hỗn hợp gồm Fe, FeO, $Fe_2O_3$ và $Fe_3O_4$ vào dung d
  [1] score=1.0: (Điều chế khí) Cho 15,8 gam $KMnO_4$ tác dụng hoàn toàn với dung dịch HCl đậm đặc, dư. Tính thể tích
  [2] score=N/A: Cho 11,0 gam hỗn hợp gồm Al và Fe tác dụng vừa đủ với dung dịch $HNO_3$ loãng, thu được 6,72 lít khí

Rejected items: 3
  [0] (Xác định kim loại) Hòa tan 4,41 gam một kim loại M bằng dung dịch $HNO_3$ dư thu được hỗn hợp khí g
      reason: Accuracy issue: The calculation in the explanation (
  [1] Cho 11,2 gam bột Fe và 6,4 gam bột Cu vào dung dịch $HNO_3$ loãng dư. Sau khi các phản ứng xảy ra ho
      reason: Kết quả tính toán trong lời giải (5,97 lít) không khớp với đáp án chọn (D. 7,47 lít). Cần kiểm tra lại các phương án trả lời.
  [2] Cho sơ đồ phản ứng: $Fe_xO_y + HNO_3 \rightarrow Fe(NO_3)_3 + NO + H_2O$. Sau

In [31]:
# ── Formatter drop analysis ──
reviewed = res_r2.get('reviewed_items', [])
print(f"Reviewed items in state: {len(reviewed)}")
print(f"Formatter trims to num_questions={req_r2.num_questions}")
print(f"Items sent to formatter: {min(len(reviewed), req_r2.num_questions)}")
print(f"Final quiz output: {len(res_r2['final_output'].content.quiz)}")
print(f"Drop rate: {min(len(reviewed), req_r2.num_questions) - len(res_r2['final_output'].content.quiz)} items lost in formatting")

# Check item structure  
if reviewed:
    keys = list(reviewed[0].keys())
    print(f"\nItem keys: {keys}")
    has_q = sum(1 for r in reviewed if 'question' in r)
    has_opts = sum(1 for r in reviewed if 'options' in r or 'choices' in r)
    print(f"Items with 'question': {has_q}/{len(reviewed)}")
    print(f"Items with 'options': {has_opts}/{len(reviewed)}")

Reviewed items in state: 5
Formatter trims to num_questions=10
Items sent to formatter: 5
Final quiz output: 3
Drop rate: 2 items lost in formatting

Item keys: ['question', 'answer', 'explanation', 'topic', 'difficulty', 'context_source', 'doc_scope', 'computation_trace', 'review_score']
Items with 'question': 5/5
Items with 'options': 0/5


In [32]:
# ── Reload formatter with retry fix ──
import importlib
import src.graph.nodes.formatter as _fmt_mod
importlib.reload(_fmt_mod)
from src.graph.nodes.formatter import formatter_node as _fmt_node

# Also re-reload builder to pick up new formatter
import src.graph.builder as _builder_mod
importlib.reload(_builder_mod)
from src.graph.builder import compile_graph as _cg_r2c
globals()['compile_graph'] = _cg_r2c

print(f"Formatter _FORMAT_MAX_RETRIES = {_fmt_mod._FORMAT_MAX_RETRIES}")
print("✅ Formatter retry fix loaded")

Formatter _FORMAT_MAX_RETRIES = 2
✅ Formatter retry fix loaded


In [33]:
# ── Round 2c: With formatter retry fix ──
import time

req_r2c = make_request(
    'Phản ứng oxi hóa khử',
    num_questions=10,
    difficulty='high_application',
    game_types=['quiz'],
)

t0 = time.time()
res_r2c = await run_pipeline(req_r2c)
t1 = time.time()

ALL['r2c_high_application'] = summarize(res_r2c, 'r2c_high_application')
s = ALL['r2c_high_application']
pr2c = s['passed'] / s['generated'] * 100 if s['generated'] > 0 else 0

print(f"\nRound 2c (formatter retry): {pr2c:.1f}% review pass rate")
print(f"  generated={s['generated']}, passed={s['passed']}, rejected={s['rejected']}")
print(f"  quiz output={s['quiz']}, iterations={s['iterations']}")
print(f"  delivery: {s['quiz']}/{req_r2c.num_questions} requested ({s['quiz']/req_r2c.num_questions*100:.0f}%)")
print(f"  time={t1-t0:.1f}s")

# Show sample
out = res_r2c.get('final_output')
if out and out.content.quiz:
    q0 = out.content.quiz[0]
    print(f"\nSample: {q0.question[:200]}")

2026-03-29 16:07:30 [info     ] creating_graph                
2026-03-29 16:07:30 [info     ] graph_created                 
2026-03-29 16:07:30 [info     ] graph_compiled_without_checkpointer
2026-03-29 16:07:30 [info     ] supervisor_analyzing           doc_scope=system topic='Phản ứng oxi hóa khử' user_id=pipeline_test
2026-03-29 16:07:32 [info     ] supervisor_classified          content_type=math
2026-03-29 16:07:32 [info     ] routing_to_agent               content_type=math route=math_agent
2026-03-29 16:07:32 [info     ] math_agent_starting            doc_scope=system iteration=0 num_questions=10 num_still_needed=10 num_to_generate=20 topic='Phản ứng oxi hóa khử' user_id=pipeline_test
2026-03-29 16:07:32 [info     ] retrieving_context             doc_scope=system query='Phản ứng oxi hóa khử high_application level educational content' user_id=pipeline_test
2026-03-29 16:07:32 [debug    ] creating_retriever             doc_scope=system extractive_answers=True filter='user_id: AN

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-29 16:07:34 [debug    ] retrieved_documents            count=30
2026-03-29 16:07:34 [info     ] context_retrieved              chunks=30 sources=30
2026-03-29 16:07:34 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-29 16:08:38 [info     ] math_agent_code_execution_done code_traces_count=6 generation_attempt=1 text_length=13639


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:1076: UserWarning: 
        - 'executable_code': Always present.
        - 'execution_result' & 'image_url': May be absent for some queries.

        Validate before using in production.

  warnings.warn(


2026-03-29 16:09:22 [error    ] math_agent_generation_error    error='Failed to parse ContentItemList from completion {"items": [{"question": "Cho 2,4 gam Mg và 2,7 gam Al tác dụng hết với dung dịch $HNO_3$ dư, thu được 0,224 lít khí $N_2$ (đktc) và 0,224 lít khí $N_2O$ (đktc). Cô cạn dung dịch sau phản ứng thu được khối lượng muối khan là bao nhiêu?", "answer": "39,3 gam", "explanation": "$n_{Mg} = 0,1; n_{Al} = 0,1 \\\\rightarrow \\\\sum n_{e\\\\, nhường} = 0,1 \\\\times 2 + 0,1 \\\\times 3 = 0,5$ mol. $n_{N_2} = 0,01; n_{N_2O} = 0,01 \\\\rightarrow \\\\sum n_{e\\\\, nhận\\\\, bởi\\\\, khí} = 0,01 \\\\times 10 + 0,01 \\\\times 8 = 0,18$ mol. Vì $0,5 > 0,18$, sản phẩm khử có $NH_4NO_3$. $n_{e\\\\, (NH_4NO_3)} = 0,5 - 0,18 = 0,32$ mol $\\\\rightarrow n_{NH_4NO_"}]}. Got: 1 validation error for ContentItemList\nitems.0.topic\n  Field required [type=missing, input_value={\'question\': \'Cho 2,4 gam...\\rightarrow n_{NH_4NO_\'}, input_type=dict]\n    For further information visit https://

...

2026-03-29 16:14:03 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-29 16:14:44 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=7
2026-03-29 16:15:18 [info     ] math_agent_batch_parsed        attempt=1 batch=3 items=6
2026-03-29 16:15:18 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=20
2026-03-29 16:15:18 [info     ] math_agent_generated           items_count=20
2026-03-29 16:15:18 [info     ] reviewer_reviewing             items_count=20 iteration=1
2026-03-29 16:15:40 [info     ] reviewer_completed             iteration=1 passed=2 rejected=1
2026-03-29 16:15:40 [info     ] review_router_deciding         iteration=2 max_iterations=3 pass_rate=0.5 rejected=2 reviewed=2
2026-03-29 16:15:40 [info     ] review_router_fail             have=2 need=7 reason=insufficient_items
2026-03-29 16:15:40 [info     ] supervisor_analyzing           doc_scope=system topic='Phản ứng oxi hóa khử' user_id=pipeline_test
2026-03-29 16:1

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-29 16:15:44 [debug    ] retrieved_documents            count=30
2026-03-29 16:15:44 [info     ] context_retrieved              chunks=30 sources=30
2026-03-29 16:15:44 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-29 16:16:46 [info     ] math_agent_code_execution_done code_traces_count=4 generation_attempt=1 text_length=7312


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:1076: UserWarning: 
        - 'executable_code': Always present.
        - 'execution_result' & 'image_url': May be absent for some queries.

        Validate before using in production.

  warnings.warn(


2026-03-29 16:17:08 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-29 16:17:49 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=1
2026-03-29 16:18:03 [info     ] math_agent_batch_parsed        attempt=1 batch=3 items=2
2026-03-29 16:18:03 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=10
2026-03-29 16:18:03 [info     ] math_agent_generated           items_count=10
2026-03-29 16:18:03 [info     ] reviewer_reviewing             items_count=10 iteration=2
2026-03-29 16:18:23 [info     ] reviewer_completed             iteration=2 passed=10 rejected=0
2026-03-29 16:18:23 [info     ] review_router_deciding         iteration=3 max_iterations=3 pass_rate=0.8571428571428571 rejected=2 reviewed=12
2026-03-29 16:18:23 [info     ] review_router_pass             reason=sufficient_items
2026-03-29 16:18:23 [info     ] formatter_trimming             after=10 before=12
2026-03-29 16:18:23 [info     ] formatter_starting             

In [34]:
# ── Round 2c results ──
s = ALL.get('r2c_high_application')
if s:
    pr = s['passed'] / s['generated'] * 100 if s['generated'] > 0 else 0
    delivery = s['quiz'] / req_r2c.num_questions * 100
    print(f"Review pass rate: {pr:.1f}%")
    print(f"Delivery rate: {s['quiz']}/{req_r2c.num_questions} = {delivery:.0f}%")
    print(f"  generated={s['generated']}, passed={s['passed']}, rejected={s['rejected']}")
    print(f"  quiz={s['quiz']}, iterations={s['iterations']}")
    
    # Compare formatter drops
    reviewed_count = len(res_r2c.get('reviewed_items', []))
    trimmed = min(reviewed_count, req_r2c.num_questions)
    dropped = trimmed - s['quiz']
    print(f"\nFormatter: reviewed={reviewed_count}, trimmed={trimmed}, quiz_out={s['quiz']}, dropped={dropped}")
    print(f"Formatter recovery rate: {s['quiz']/trimmed*100:.0f}%" if trimmed else "N/A")

Review pass rate: 83.3%
Delivery rate: 10/10 = 100%
  generated=12, passed=10, rejected=2
  quiz=10, iterations=3

Formatter: reviewed=12, trimmed=10, quiz_out=10, dropped=0
Formatter recovery rate: 100%


In [35]:
# ── Feedback classification analysis ──
from src.graph.nodes.math_agent import _classify_feedback

rejected = res_r2c.get('rejected_items', [])
print(f"Rejected items: {len(rejected)}\n")

categories = {}
for i, item in enumerate(rejected):
    feedback = item.get('review_feedback', '')
    category = _classify_feedback(feedback)
    categories[category] = categories.get(category, 0) + 1
    q = item.get('question', '')[:120]
    print(f"[{i}] Category: {category}")
    print(f"    Q: {q}")
    print(f"    Feedback: {feedback[:200]}")
    print()

print("Category distribution:")
for cat, count in sorted(categories.items(), key=lambda x: -x[1]):
    print(f"  {cat}: {count}")

Rejected items: 2

[0] Category: other
    Q: Hòa tan hoàn toàn 19,2 gam kim loại M trong dung dịch $HNO_3$ dư thu được 4,958 lít hỗn hợp khí X (đkc) gồm $NO$ và $N_2
    Feedback: The question is mathematically inconsistent. With the given volume (4.958L đkc = 0.2 mol) and density (M=37), the moles of electron transferred is 1.1 mol. For 19.2g of metal M to transfer 1.1 mol of 

[1] Category: other
    Q: Cho 15,8 gam $KMnO_4$ tác dụng hoàn toàn với dung dịch $HCl$ đậm đặc, dư. Khí $Cl_2$ sinh ra được dẫn vào dung dịch $KOH
    Feedback: The

Category distribution:
  other: 2


In [36]:
# ── Reload math_agent with updated accuracy pattern ──
import importlib
import src.graph.nodes.math_agent as _ma_mod
importlib.reload(_ma_mod)
import src.graph.builder as _b_mod
importlib.reload(_b_mod)
from src.graph.builder import compile_graph as _cg_r2d
globals()['compile_graph'] = _cg_r2d
print("✅ Reloaded")

✅ Reloaded


In [37]:
# ── Round 2d: Broader difficulty level test (all 4 levels) ──
import time

TOPICS_BY_DIFF = {
    'recall':           'Phản ứng oxi hóa khử',
    'comprehension':    'Phản ứng oxi hóa khử',
    'application':      'Phản ứng oxi hóa khử',
    'high_application': 'Phản ứng oxi hóa khử',
}

for diff, topic in TOPICS_BY_DIFF.items():
    key = f'r2d_{diff}'
    if key in ALL:
        print(f'⏭ {key} already run, skipping')
        continue
    print(f'\n{"="*60}')
    print(f'Running: {diff} (topic={topic})')
    print(f'{"="*60}')
    req = make_request(topic, num_questions=10, game_types=['quiz'], difficulty=diff)
    t0 = time.time()
    res = await run_pipeline(req)
    t1 = time.time()
    ALL[key] = summarize(res, key)
    s = ALL[key]
    if s:
        pr = s['passed'] / s['generated'] * 100 if s['generated'] > 0 else 0
        print(f'  → pass_rate={pr:.1f}%, delivery={s["quiz"]}/10 ({s["quiz"]*10}%)')
    print(f'  → time={t1-t0:.1f}s')

# ── Summary table ──
print(f'\n{"="*60}')
print(f'{"Difficulty":<20} {"PassRate":>10} {"Delivery":>10} {"Gen":>5} {"Pass":>5} {"Rej":>5} {"Quiz":>5} {"Time":>8}')
print(f'{"-"*20} {"-"*10} {"-"*10} {"-"*5} {"-"*5} {"-"*5} {"-"*5} {"-"*8}')
for diff in TOPICS_BY_DIFF:
    key = f'r2d_{diff}'
    s = ALL.get(key)
    if not s:
        print(f'{diff:<20} {"FAILED":>10}')
        continue
    pr = s['passed'] / s['generated'] * 100 if s['generated'] > 0 else 0
    dlv = f"{s['quiz']}/10"
    print(f'{diff:<20} {pr:>9.1f}% {dlv:>10} {s["generated"]:>5} {s["passed"]:>5} {s["rejected"]:>5} {s["quiz"]:>5} {s["elapsed"]:>7.1f}s')

# Overall
vals = [ALL[f'r2d_{d}'] for d in TOPICS_BY_DIFF if ALL.get(f'r2d_{d}')]
if vals:
    avg_pr = sum(v['passed']/v['generated']*100 if v['generated']>0 else 0 for v in vals) / len(vals)
    avg_dlv = sum(v['quiz'] for v in vals) / (len(vals)*10) * 100
    print(f'\nOverall: avg_pass_rate={avg_pr:.1f}%, avg_delivery={avg_dlv:.0f}%')


Running: recall (topic=Phản ứng oxi hóa khử)
2026-03-29 16:24:47 [info     ] creating_graph                
2026-03-29 16:24:47 [info     ] graph_created                 
2026-03-29 16:24:47 [info     ] graph_compiled_without_checkpointer
2026-03-29 16:24:47 [info     ] supervisor_analyzing           doc_scope=system topic='Phản ứng oxi hóa khử' user_id=pipeline_test
2026-03-29 16:24:49 [info     ] supervisor_classified          content_type=math
2026-03-29 16:24:49 [info     ] routing_to_agent               content_type=math route=math_agent
2026-03-29 16:24:49 [info     ] math_agent_starting            doc_scope=system iteration=0 num_questions=10 num_still_needed=10 num_to_generate=11 topic='Phản ứng oxi hóa khử' user_id=pipeline_test
2026-03-29 16:24:49 [info     ] retrieving_context             doc_scope=system query='Phản ứng oxi hóa khử recall level educational content' user_id=pipeline_test
2026-03-29 16:24:49 [debug    ] creating_retriever             doc_scope=system extract

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-29 16:24:51 [debug    ] retrieved_documents            count=30
2026-03-29 16:24:51 [info     ] context_retrieved              chunks=30 sources=30
2026-03-29 16:24:51 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-29 16:25:08 [info     ] math_agent_code_execution_done code_traces_count=2 generation_attempt=1 text_length=3898


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:1076: UserWarning: 
        - 'executable_code': Always present.
        - 'execution_result' & 'image_url': May be absent for some queries.

        Validate before using in production.

  warnings.warn(


2026-03-29 16:25:21 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-29 16:25:41 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=4
2026-03-29 16:25:41 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=11
2026-03-29 16:25:41 [info     ] math_agent_generated           items_count=11
2026-03-29 16:25:41 [info     ] reviewer_reviewing             items_count=11 iteration=0
2026-03-29 16:25:54 [info     ] reviewer_completed             iteration=0 passed=11 rejected=0
2026-03-29 16:25:54 [info     ] review_router_deciding         iteration=1 max_iterations=3 pass_rate=1.0 rejected=0 reviewed=11
2026-03-29 16:25:54 [info     ] review_router_pass             reason=sufficient_items
2026-03-29 16:25:54 [info     ] formatter_trimming             after=10 before=11
2026-03-29 16:25:54 [info     ] formatter_starting             game_types=['quiz'] items_count=10 user_id=pipeline_test
2026-03-29 16:26:04 [info     ] formatter_quiz_

...

2026-03-29 16:36:11 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-29 16:36:50 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=7
2026-03-29 16:37:09 [info     ] math_agent_batch_parsed        attempt=1 batch=3 items=6
2026-03-29 16:37:09 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=20
2026-03-29 16:37:09 [info     ] math_agent_generated           items_count=20
2026-03-29 16:37:09 [info     ] reviewer_reviewing             items_count=20 iteration=1
2026-03-29 16:37:31 [info     ] reviewer_completed             iteration=1 passed=3 rejected=0
2026-03-29 16:37:31 [info     ] review_router_deciding         iteration=2 max_iterations=3 pass_rate=0.75 rejected=1 reviewed=3
2026-03-29 16:37:31 [info     ] review_router_fail             have=3 need=7 reason=insufficient_items
2026-03-29 16:37:31 [info     ] supervisor_analyzing           doc_scope=system topic='Phản ứng oxi hóa khử' user_id=pipeline_test
2026-03-29 16:

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-29 16:37:36 [debug    ] retrieved_documents            count=30
2026-03-29 16:37:36 [info     ] context_retrieved              chunks=30 sources=30
2026-03-29 16:37:36 [info     ] math_agent_context_retrieved   context_chunks=30
2026-03-29 16:38:40 [info     ] math_agent_code_execution_done code_traces_count=3 generation_attempt=1 text_length=8039


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:1076: UserWarning: 
        - 'executable_code': Always present.
        - 'execution_result' & 'image_url': May be absent for some queries.

        Validate before using in production.

  warnings.warn(


2026-03-29 16:39:14 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-03-29 16:39:55 [error    ] math_agent_generation_error    error='Failed to parse ContentItemList from completion {"items": [{"question": "Cho phản ứng: $CH_3-CH=CH_2 + KMnO_4 + H_2O \\\\rightarrow CH_3-CH(OH)-CH_2(OH) + MnO_2 + KOH$. Tổng hệ số (nguyên, tối giản) của các chất trong phương trình hóa học là:", "answer": "D. 16.", "explanation": "Cân bằng phản ứng oxi hóa - khử hữu cơ theo phương pháp thăng bằng electron: $3C_3H_6 + 2KMnO_4 + 4H_2O \\\\rightarrow 3C_3H_6(OH)_2 + 2MnO_2 + 2KOH$. Tổng hệ số các chất tham gia và sản phẩm là: $3 + 2 + 4 + 3 + 2 + 2 = 16$.", "topic": "Phản ứng oxi hóa khử hữu cơ", "computation_trace": null}, {"question": "Để điều chế $HNO_3$ trong công nghiệp, người ta oxi hóa $NH_3$ bằng oxi không khí có xúc tác Pt. Nếu dùng 170 kg $NH_3$ và hiệu suất toàn bộ quá trình là 80%, thì khối lượng dung dịch $HNO_3$ 63% thu được là:"}]}. Got: 3 validation errors for Content

In [38]:
# ── Round 2d results summary ──
print(f'{"Difficulty":<20} {"PassRate":>10} {"Delivery":>10} {"Gen":>5} {"Pass":>5} {"Rej":>5} {"Quiz":>5} {"Iter":>5} {"Time":>8}')
print(f'{"-"*20} {"-"*10} {"-"*10} {"-"*5} {"-"*5} {"-"*5} {"-"*5} {"-"*5} {"-"*8}')
for diff in ['recall', 'comprehension', 'application', 'high_application']:
    key = f'r2d_{diff}'
    s = ALL.get(key)
    if not s:
        print(f'{diff:<20} {"FAILED":>10}')
        continue
    pr = s['passed'] / s['generated'] * 100 if s['generated'] > 0 else 0
    dlv = f"{s['quiz']}/10"
    print(f'{diff:<20} {pr:>9.1f}% {dlv:>10} {s["generated"]:>5} {s["passed"]:>5} {s["rejected"]:>5} {s["quiz"]:>5} {s["iterations"]:>5} {s["elapsed"]:>7.1f}s')

vals = [ALL[f'r2d_{d}'] for d in ['recall','comprehension','application','high_application'] if ALL.get(f'r2d_{d}')]
if vals:
    avg_pr = sum(v['passed']/v['generated']*100 if v['generated']>0 else 0 for v in vals) / len(vals)
    avg_dlv = sum(v['quiz'] for v in vals) / (len(vals)*10) * 100
    total_time = sum(v['elapsed'] for v in vals)
    print(f'\nOverall: avg_pass_rate={avg_pr:.1f}%, avg_delivery={avg_dlv:.0f}%, total_time={total_time:.0f}s')

Difficulty             PassRate   Delivery   Gen  Pass   Rej  Quiz  Iter     Time
-------------------- ---------- ---------- ----- ----- ----- ----- ----- --------
recall                   100.0%      10/10    10    10     0    10     1    84.6s
comprehension             88.9%       8/10     9     8     1     8     1   149.7s
application              100.0%      10/10    10    10     0    10     2   233.5s
high_application          60.0%       3/10     5     3     2     3     3   584.3s

Overall: avg_pass_rate=87.2%, avg_delivery=78%, total_time=1052s


In [39]:
# ── Round 2d high_application analysis ──
# Check why only 5 generated in 3 iterations
# Dig into the last run result
ha_key = 'r2d_high_application'
s = ALL.get(ha_key)
print(f"high_application: generated={s['generated']}, passed={s['passed']}, rejected={s['rejected']}, iterations={s['iterations']}")
print(f"Only 5 generated means pipeline struggled to produce enough items\n")

# Check all run keys
print("All round 2d keys in ALL:")
for k in sorted(ALL.keys()):
    if k.startswith('r2d_'):
        v = ALL[k]
        pr = v['passed']/v['generated']*100 if v['generated']>0 else 0
        print(f"  {k}: gen={v['generated']} pass={v['passed']} rej={v['rejected']} quiz={v['quiz']} iter={v['iterations']} pr={pr:.0f}%")

# Compare with Round 2c
r2c = ALL.get('r2c_high_application')
if r2c:
    pr2c = r2c['passed']/r2c['generated']*100 if r2c['generated']>0 else 0
    print(f"\nRound 2c: gen={r2c['generated']} pass={r2c['passed']} rej={r2c['rejected']} quiz={r2c['quiz']} iter={r2c['iterations']} pr={pr2c:.0f}%")
    print(f"Round 2d: gen={s['generated']} pass={s['passed']} rej={s['rejected']} quiz={s['quiz']} iter={s['iterations']} pr={s['passed']/s['generated']*100:.0f}%")
    print(f"\nHigh variance: 2c got {r2c['generated']} items generated vs 2d got {s['generated']}")
    print(f"This is inherent model variability — pass rate 60-83%, delivery 30-100%")

high_application: generated=5, passed=3, rejected=2, iterations=3
Only 5 generated means pipeline struggled to produce enough items

All round 2d keys in ALL:
  r2d_application: gen=10 pass=10 rej=0 quiz=10 iter=2 pr=100%
  r2d_comprehension: gen=9 pass=8 rej=1 quiz=8 iter=1 pr=89%
  r2d_high_application: gen=5 pass=3 rej=2 quiz=3 iter=3 pr=60%
  r2d_recall: gen=10 pass=10 rej=0 quiz=10 iter=1 pr=100%

Round 2c: gen=12 pass=10 rej=2 quiz=10 iter=3 pr=83%
Round 2d: gen=5 pass=3 rej=2 quiz=3 iter=3 pr=60%

High variance: 2c got 12 items generated vs 2d got 5
This is inherent model variability — pass rate 60-83%, delivery 30-100%
